# Investigating the behaviour of entrapped databases 

This notebook studies the properties of entrapped databases. Here are some of the definitions:
- Proteins ending in `_target` represent the **original target protein database**.
- Proteins ending in `_p_target` represent the **entrapped protein database**.
- Peptides linked to `_target` are treated as **original peptides**.
- Peptides linked to `_p_target` are treated as **entrapped peptides**.

This notebook is written for data table generated using FDRBench, and assumes that there would be at least three columns:

| column | meaning |
|---|---|
| `sequence` | peptide sequence |
| `decoy` | decoy label, for example `Yes`/`No`, `True`/`False`, or similar |
| `proteins` | semicolon-separated protein accessions|

The aim is to check whether the entrapped peptide database preserves important peptide level and protein-level properties of the original database, while remaining distinguishable enough to support FDR experiments.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import os
from pathlib import Path
from collections import Counter

try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
    warnings.warn("scipy is not available; statistical tests will be skipped.")

PROJECT_DIR = Path().home()/"Desktop"/"analysis"/"spectral-lir"
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 14

# Sanity check: print the current working directory
os.chdir(PROJECT_DIR)
print("Current working directory:", Path.cwd())

OUTPUT_DIR = PROJECT_DIR/"results"

## Randomly shuffled and foreign species (*C. elegans*) entrapped database at the peptide-level

In [ ]:
# Define path to entrapped data
# rs means "randomly shuffled"
# fs means "foreign species"
database_rs_path = "data/reference-fasta/UP000000589_10090_mouse_20241210_diann_entrapment_pep.txt"
database_fs_path = "data/reference-fasta/UP000000589_foreign_species_C_elegans_entrapment_pep.txt"
df_diann_rs = pd.read_csv(database_rs_path, sep="\t")
df_diann_fs = pd.read_csv(database_fs_path, sep="\t")

# Check the first few rows of the DataFrame
print(df_diann_rs.head())
print(df_diann_fs.head())

### Classify original, entrapped, decoy, and mixed protein groups

At the peptide-level entrapped database, the .txt file contains a column `peptide_type` which can be used to distinguish between `target` and `entrapped` peptides.

In [ ]:
# Primary classification logic: use peptide_type directly.
df_diann_rs["database_class"] = df_diann_rs["peptide_type"].map({
    "target": "target",
    "p_target": "entrapped",
})

df_diann_rs["is_target"] = df_diann_rs["peptide_type"].eq("target")
df_diann_rs["is_entrapped"] = df_diann_rs["peptide_type"].eq("p_target")

# Decoy parsing.
true_like = {"yes", "y", "true", "t", "1", "decoy"}
df_diann_rs["is_decoy"] = df_diann_rs["decoy"].astype(str).str.strip().str.lower().isin(true_like)

# Basic sequence/protein-group annotations.
df_diann_rs["peptide_length"] = df_diann_rs["sequence"].str.len()
df_diann_rs["n_proteins"] = df_diann_rs["proteins"].str.split(";").map(lambda x: len([p for p in x if str(p).strip()]))

summary = (
    df_diann_rs.groupby("database_class")
    .agg(
        n_rows=("sequence", "size"),
        n_unique_sequences=("sequence", "nunique"),
        n_unique_protein_groups=("proteins", "nunique"),
        mean_length=("peptide_length", "mean"),
        median_length=("peptide_length", "median"),
        mean_n_proteins=("n_proteins", "mean"),
        decoy_fraction=("is_decoy", "mean"),
    )
    .round(4)
)
summary

In [ ]:
# Primary classification logic: use peptide_type directly.
df_diann_fs["database_class"] = df_diann_fs["peptide_type"].map({
    "target": "target",
    "p_target": "entrapped",
})

df_diann_fs["is_target"] = df_diann_fs["peptide_type"].eq("target")
df_diann_fs["is_entrapped"] = df_diann_fs["peptide_type"].eq("p_target")

# Decoy parsing.
true_like = {"yes", "y", "true", "t", "1", "decoy"}
df_diann_fs["is_decoy"] = df_diann_fs["decoy"].astype(str).str.strip().str.lower().isin(true_like)

# Basic sequence/protein-group annotations.
df_diann_fs["peptide_length"] = df_diann_fs["sequence"].str.len()
df_diann_fs["n_proteins"] = df_diann_fs["proteins"].str.split(";").map(lambda x: len([p for p in x if str(p).strip()]))

summary = (
    df_diann_fs.groupby("database_class")
    .agg(
        n_rows=("sequence", "size"),
        n_unique_sequences=("sequence", "nunique"),
        n_unique_protein_groups=("proteins", "nunique"),
        mean_length=("peptide_length", "mean"),
        median_length=("peptide_length", "median"),
        mean_n_proteins=("n_proteins", "mean"),
        decoy_fraction=("is_decoy", "mean"),
    )
    .round(4)
)
summary

### Peptide length distribution check

In [ ]:
plot_df = df_diann_rs[df_diann_rs["database_class"].isin(["target", "entrapped"])].copy()

fig, ax = plt.subplots(figsize=(8, 5))
for label, group in plot_df.groupby("database_class"):
    ax.hist(
        group["peptide_length"],
        bins=range(0, int(plot_df["peptide_length"].max()) + 2),
        alpha=0.9,
        density=True,
        label=label,
    )
ax.set_xlabel("Peptide length")
ax.set_ylabel("Density")
ax.set_xlim(5, 40)
ax.set_title("Peptide length distribution - randomly shuffled sequences")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "peptide_length_distribution_randomly_shuffled.svg", dpi=200)
plt.show()

In [ ]:
# Primary classification logic: use peptide_type directly.
df_diann_fs["database_class"] = df_diann_fs["peptide_type"].map({
    "target": "target",
    "p_target": "entrapped",
})

df_diann_fs["is_target"] = df_diann_fs["peptide_type"].eq("target")
df_diann_fs["is_entrapped"] = df_diann_fs["peptide_type"].eq("p_target")

# Decoy parsing.
true_like = {"yes", "y", "true", "t", "1", "decoy"}
df_diann_fs["is_decoy"] = df_diann_fs["decoy"].astype(str).str.strip().str.lower().isin(true_like)

# Basic sequence/protein-group annotations.
df_diann_fs["peptide_length"] = df_diann_fs["sequence"].str.len()
df_diann_fs["n_proteins"] = df_diann_fs["proteins"].str.split(";").map(lambda x: len([p for p in x if str(p).strip()]))

summary = (
    df_diann_fs.groupby("database_class")
    .agg(
        n_rows=("sequence", "size"),
        n_unique_sequences=("sequence", "nunique"),
        n_unique_protein_groups=("proteins", "nunique"),
        mean_length=("peptide_length", "mean"),
        median_length=("peptide_length", "median"),
        mean_n_proteins=("n_proteins", "mean"),
        decoy_fraction=("is_decoy", "mean"),
    )
    .round(4)
)

summary

plot_df = df_diann_fs[df_diann_fs["database_class"].isin(["target", "entrapped"])].copy()

fig, ax = plt.subplots(figsize=(8, 5))
for label, group in plot_df.groupby("database_class"):
    ax.hist(
        group["peptide_length"],
        bins=range(0, int(plot_df["peptide_length"].max()) + 2),
        alpha=0.9,
        density=True,
        label=label,
    )
ax.set_xlabel("Peptide length")
ax.set_ylabel("Density")
ax.set_xlim(5, 40)
ax.set_title("Peptide length distribution - foreign species entrapped sequences")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "peptide_length_distribution_foreign_species_entrapped.svg", dpi=200)
plt.show()

### Peptide physicochemical properties

These properties are calculated directly from peptide sequences:
- length
- monoisotopic mass approximation
- hydrophobicity using Kyte-Doolittle scores
- aromatic, basic, and acidic residue fractions
- missed-cleavage proxy: internal K/R not followed by P
- terminal trypticity proxy

In [ ]:
AA_MASS = {
    "A": 71.03711, "R": 156.10111, "N": 114.04293, "D": 115.02694, "C": 103.00919,
    "E": 129.04259, "Q": 128.05858, "G": 57.02146, "H": 137.05891, "I": 113.08406,
    "L": 113.08406, "K": 128.09496, "M": 131.04049, "F": 147.06841, "P": 97.05276,
    "S": 87.03203, "T": 101.04768, "W": 186.07931, "Y": 163.06333, "V": 99.06841
}
WATER = 18.01056

KD = {
    "I": 4.5, "V": 4.2, "L": 3.8, "F": 2.8, "C": 2.5, "M": 1.9, "A": 1.8,
    "G": -0.4, "T": -0.7, "S": -0.8, "W": -0.9, "Y": -1.3, "P": -1.6,
    "H": -3.2, "E": -3.5, "Q": -3.5, "D": -3.5, "N": -3.5, "K": -3.9, "R": -4.5
}

VALID_AA = set(AA_MASS)

def clean_sequence(seq: str) -> str:
    return "".join([aa for aa in str(seq).upper() if aa in VALID_AA])

def peptide_mass(seq: str) -> float:
    seq = clean_sequence(seq)
    return WATER + sum(AA_MASS[aa] for aa in seq)

def gravy(seq: str) -> float:
    seq = clean_sequence(seq)
    return np.nan if len(seq) == 0 else np.mean([KD[aa] for aa in seq])

def fraction_of(seq: str, aas: str) -> float:
    seq = clean_sequence(seq)
    aa_set = set(aas)
    return np.nan if len(seq) == 0 else sum(aa in aa_set for aa in seq) / len(seq)

def missed_cleavage_proxy(seq: str) -> int:
    seq = clean_sequence(seq)
    return sum(seq[i] in {"K", "R"} and seq[i + 1] != "P" for i in range(max(0, len(seq) - 1)))

def c_terminal_tryptic(seq: str) -> bool:
    seq = clean_sequence(seq)
    return len(seq) > 0 and seq[-1] in {"K", "R"}

#### For randomly shuffled database

In [ ]:
df_diann_rs["clean_sequence"] = df_diann_rs["sequence"].map(clean_sequence)
df_diann_rs["mass"] = df_diann_rs["sequence"].map(peptide_mass)
df_diann_rs["gravy"] = df_diann_rs["sequence"].map(gravy)
df_diann_rs["aromatic_fraction"] = df_diann_rs["sequence"].map(lambda s: fraction_of(s, "FWY"))
df_diann_rs["basic_fraction"] = df_diann_rs["sequence"].map(lambda s: fraction_of(s, "KRH"))
df_diann_rs["acidic_fraction"] = df_diann_rs["sequence"].map(lambda s: fraction_of(s, "DE"))
df_diann_rs["missed_cleavage_proxy"] = df_diann_rs["sequence"].map(missed_cleavage_proxy)
df_diann_rs["c_terminal_tryptic"] = df_diann_rs["sequence"].map(c_terminal_tryptic)

property_cols = [
    "peptide_length", "mass", "gravy", "aromatic_fraction", "basic_fraction",
    "acidic_fraction", "missed_cleavage_proxy", "c_terminal_tryptic"
]

# Use peptide_type directly for classification
df_diann_rs["database_class"] = df_diann_rs["peptide_type"].map({
    "target": "target",
    "p_target": "entrapped"
})

plot_df = df_diann_rs[df_diann_rs["database_class"].notna()].copy()

plot_df[
    [
        "sequence",
        "peptide_type",
        "database_class",
        "peptide_length",
        "mass",
        "gravy",
        "aromatic_fraction",
        "basic_fraction",
        "acidic_fraction",
        "missed_cleavage_proxy",
    ]
].head()

property_summary = df_diann_rs.groupby("database_class")[property_cols].agg(["mean", "median", "std"]).round(4)
property_summary


In [ ]:
def two_sample_tests(df, cols, class_col="database_class", a="target", b="entrapped"):
    rows = []
    for col in cols:
        x = df.loc[df[class_col] == a, col].dropna()
        y = df.loc[df[class_col] == b, col].dropna()
        row = {
            "property": col,
            "n_target": len(x),
            "n_entrapped": len(y),
            "mean_target": x.mean(),
            "mean_entrapped": y.mean(),
            "delta_mean_entrapped_minus_target": y.mean() - x.mean(),
        }
        if SCIPY_AVAILABLE and len(x) > 1 and len(y) > 1:
            row["ks_stat"], row["ks_p"] = stats.ks_2samp(x, y)
            row["mannwhitney_u"], row["mannwhitney_p"] = stats.mannwhitneyu(x, y, alternative="two-sided")
        rows.append(row)
    return pd.DataFrame(rows)

tests = two_sample_tests(df_diann_rs, property_cols)
tests.round(5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()

cols = ["mass", "gravy", "aromatic_fraction", "basic_fraction", "acidic_fraction", "missed_cleavage_proxy"]

for ax, col in zip(axes, cols):
    for label, group in plot_df.groupby("database_class"):
        ax.hist(group[col].dropna(), bins=40, alpha=1.0, density=True, label=label)
    ax.set_xlabel(col)
    ax.set_ylabel("Density")
    ax.set_title(col)

# Only show one legend for the whole figure (avoids 6 repeated legends)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.02))

fig.suptitle("Distributions of physicochemical properties - randomly shuffled sequences", y=1.06)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "distributions_grid_randomly_shuffled.svg", dpi=200, bbox_inches="tight")
plt.show()

#### For foreign species entrapped database

In [ ]:
df_diann_fs["clean_sequence"] = df_diann_fs["sequence"].map(clean_sequence)
df_diann_fs["mass"] = df_diann_fs["sequence"].map(peptide_mass)
df_diann_fs["gravy"] = df_diann_fs["sequence"].map(gravy)
df_diann_fs["aromatic_fraction"] = df_diann_fs["sequence"].map(lambda s: fraction_of(s, "FWY"))
df_diann_fs["basic_fraction"] = df_diann_fs["sequence"].map(lambda s: fraction_of(s, "KRH"))
df_diann_fs["acidic_fraction"] = df_diann_fs["sequence"].map(lambda s: fraction_of(s, "DE"))
df_diann_fs["missed_cleavage_proxy"] = df_diann_fs["sequence"].map(missed_cleavage_proxy)
df_diann_fs["c_terminal_tryptic"] = df_diann_fs["sequence"].map(c_terminal_tryptic)

property_cols = [
    "peptide_length", "mass", "gravy", "aromatic_fraction", "basic_fraction",
    "acidic_fraction", "missed_cleavage_proxy", "c_terminal_tryptic"
]

# Use peptide_type directly for classification
df_diann_fs["database_class"] = df_diann_fs["peptide_type"].map({
    "target": "target",
    "p_target": "entrapped"
})

plot_df = df_diann_fs[df_diann_fs["database_class"].notna()].copy()

plot_df[
    [
        "sequence",
        "peptide_type",
        "database_class",
        "peptide_length",
        "mass",
        "gravy",
        "aromatic_fraction",
        "basic_fraction",
        "acidic_fraction",
        "missed_cleavage_proxy",
    ]
].head()

property_summary = df_diann_fs.groupby("database_class")[property_cols].agg(["mean", "median", "std"]).round(4)
property_summary

tests = two_sample_tests(df_diann_fs, property_cols)
tests.round(5)

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()

cols = ["mass", "gravy", "aromatic_fraction", "basic_fraction", "acidic_fraction", "missed_cleavage_proxy"]

for ax, col in zip(axes, cols):
    for label, group in plot_df.groupby("database_class"):
        ax.hist(group[col].dropna(), bins=40, alpha=1.0, density=True, label=label)
    ax.set_xlabel(col)
    ax.set_ylabel("Density")
    ax.set_title(col)

# Only show one legend for the whole figure (avoids 6 repeated legends)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.02))

fig.suptitle("Distributions of physicochemical properties - foreign species entrapped", y=1.06)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "distributions_grid_foreign_species_entrapped.svg", dpi=200, bbox_inches="tight")
plt.show()

### Amino acid composition

#### Randomly shuffled sequences

In [ ]:
def aa_counts(sequences):
    counts = Counter()
    total = 0
    for seq in sequences:
        seq = clean_sequence(seq)
        counts.update(seq)
        total += len(seq)
    return pd.Series({aa: counts.get(aa, 0) / total if total else np.nan for aa in sorted(VALID_AA)})

aa_target = aa_counts(df_diann_rs.loc[df_diann_rs["database_class"] == "target", "sequence"])
aa_entrapped = aa_counts(df_diann_rs.loc[df_diann_rs["database_class"] == "entrapped", "sequence"])

aa_comp = pd.concat([aa_target.rename("target"), aa_entrapped.rename("entrapped")], axis=1)
aa_comp["entrapped_minus_target"] = aa_comp["entrapped"] - aa_comp["target"]
aa_comp["relative_change"] = aa_comp["entrapped_minus_target"] / aa_comp["target"].replace(0, np.nan)

aa_comp.sort_values("entrapped_minus_target", key=lambda s: s.abs(), ascending=False).round(5)

fig, ax = plt.subplots(figsize=(10, 4.5))
aa_comp[["target", "entrapped"]].plot(kind="bar", ax=ax)
ax.set_xlabel("Amino acid")
ax.set_ylabel("Residue frequency")
ax.set_title("Amino-acid composition - randomly shuffled sequences")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "amino_acid_composition_randomly_shuffled.svg", dpi=200)
plt.show()

#### Foreign species entrapped database

In [ ]:
def aa_counts(sequences):
    counts = Counter()
    total = 0
    for seq in sequences:
        seq = clean_sequence(seq)
        counts.update(seq)
        total += len(seq)
    return pd.Series({aa: counts.get(aa, 0) / total if total else np.nan for aa in sorted(VALID_AA)})

aa_target = aa_counts(df_diann_fs.loc[df_diann_fs["database_class"] == "target", "sequence"])
aa_entrapped = aa_counts(df_diann_fs.loc[df_diann_fs["database_class"] == "entrapped", "sequence"])

aa_comp = pd.concat([aa_target.rename("target"), aa_entrapped.rename("entrapped")], axis=1)
aa_comp["entrapped_minus_target"] = aa_comp["entrapped"] - aa_comp["target"]
aa_comp["relative_change"] = aa_comp["entrapped_minus_target"] / aa_comp["target"].replace(0, np.nan)

aa_comp.sort_values("entrapped_minus_target", key=lambda s: s.abs(), ascending=False).round(5)

fig, ax = plt.subplots(figsize=(10, 4.5))
aa_comp[["target", "entrapped"]].plot(kind="bar", ax=ax)
ax.set_xlabel("Amino acid")
ax.set_ylabel("Residue frequency")
ax.set_title("Amino-acid composition - foreign species entrapped")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "amino_acid_composition_foreign_species_entrapped.svg", dpi=200)
plt.show()

### Pair target and entrapped peptides using `peptide_pair_index`

This is only applicable for randomly shuffled sequences where for each target peptide, there exists a paired peptide as entrapped sequence.

In [ ]:
pair_counts = (
    df_diann_rs.groupby(["peptide_pair_index", "peptide_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["target", "p_target"]:
    if col not in pair_counts.columns:
        pair_counts[col] = 0

pair_counts["pairing_status"] = np.select(
    [
        (pair_counts["target"] == 1) & (pair_counts["p_target"] == 1),
        (pair_counts["target"] > 1) | (pair_counts["p_target"] > 1),
        (pair_counts["target"] == 0) & (pair_counts["p_target"] > 0),
        (pair_counts["target"] > 0) & (pair_counts["p_target"] == 0),
    ],
    ["one_to_one", "duplicate_rows", "entrapped_only", "target_only"],
    default="other",
)

print("Pairing status by peptide_pair_index")
display(pair_counts["pairing_status"].value_counts().to_frame("n_pair_indices"))
display(pair_counts.head(20))

paired = (
    df_diann_rs.pivot_table(
        index="peptide_pair_index",
        columns="peptide_type",
        values="sequence",
        aggfunc="first",
    )
    .reset_index()
    .rename(columns={"target": "target_sequence", "p_target": "entrapped_sequence"})
)

paired_proteins = (
    df_diann_rs.pivot_table(
        index="peptide_pair_index",
        columns="peptide_type",
        values="proteins",
        aggfunc="first",
    )
    .reset_index()
    .rename(columns={"target": "target_proteins", "p_target": "entrapped_proteins"})
)

paired = paired.merge(pair_counts[["peptide_pair_index", "pairing_status"]], on="peptide_pair_index", how="left")
paired = paired.merge(paired_proteins, on="peptide_pair_index", how="left")

one_to_one = paired[paired["pairing_status"].eq("one_to_one")].dropna(
    subset=["target_sequence", "entrapped_sequence"]
).copy()

one_to_one["target_length"] = one_to_one["target_sequence"].str.len()
one_to_one["entrapped_length"] = one_to_one["entrapped_sequence"].str.len()
one_to_one["length_delta"] = one_to_one["entrapped_length"] - one_to_one["target_length"]

print(f"One-to-one peptide pairs: {len(one_to_one):,}")
display(one_to_one.head(20))

### Sequence similarity between paired target and entrapped peptides

This section measures how different entrapped peptide is from the original peptide. Again this works the best for randomly shuffled sequences as we are looking into peptide pairs.

In [ ]:
def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current = [i]
        for j, cb in enumerate(b, start=1):
            insert = current[j - 1] + 1
            delete = previous[j] + 1
            subst = previous[j - 1] + (ca != cb)
            current.append(min(insert, delete, subst))
        previous = current
    return previous[-1]

def kmers(seq: str, k: int):
    seq = clean_sequence(seq)
    if len(seq) < k:
        return set()
    return {seq[i:i + k] for i in range(len(seq) - k + 1)}

def shared_kmer_fraction(a: str, b: str, k: int = 3) -> float:
    ka, kb = kmers(a, k), kmers(b, k)
    union = ka | kb
    return np.nan if not union else len(ka & kb) / len(union)

if len(one_to_one) > 0:
    one_to_one["exact_match"] = one_to_one["target_sequence"].eq(one_to_one["entrapped_sequence"])
    one_to_one["edit_distance"] = [
        levenshtein(a, b) for a, b in zip(one_to_one["target_sequence"], one_to_one["entrapped_sequence"])
    ]
    one_to_one["normalized_edit_distance"] = one_to_one["edit_distance"] / one_to_one[["target_length", "entrapped_length"]].max(axis=1)
    one_to_one["shared_3mer_fraction"] = [
        shared_kmer_fraction(a, b, k=3) for a, b in zip(one_to_one["target_sequence"], one_to_one["entrapped_sequence"])
    ]
    one_to_one["same_n_term"] = one_to_one["target_sequence"].str[0].eq(one_to_one["entrapped_sequence"].str[0])
    one_to_one["same_c_term"] = one_to_one["target_sequence"].str[-1].eq(one_to_one["entrapped_sequence"].str[-1])

    similarity_summary = one_to_one[[
        "length_delta", "exact_match", "edit_distance", "normalized_edit_distance",
        "shared_3mer_fraction", "same_n_term", "same_c_term"
    ]].describe(include="all")
    display(similarity_summary)
else:
    print("No one-to-one pairs found. Check peptide_pair_index and peptide_type values.")


if len(one_to_one) > 0:
    for col in ["length_delta", "edit_distance", "normalized_edit_distance", "shared_3mer_fraction"]:
        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.hist(one_to_one[col].dropna(), bins=40)
        ax.set_xlabel(col)
        ax.set_ylabel("Number of pairs")
        ax.set_title(f"Paired target vs entrapped: {col}")
        plt.tight_layout()
        # plt.savefig(OUTPUT_DIR / f"paired_{col}.png", dpi=200)
        plt.show()